In [1]:
import sys
import os

# Adjust the path to where the src folder is located
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

In [2]:
import torch.nn as nn
import torch.utils.data.dataloader
from torch.nn.functional import scaled_dot_product_attention
import numpy as np
from language_models.utils import repackage_hidden, get_batch, batchify, save_checkpoint, move_to_device, save_val_loss_data
from language_models.dictionary_corpus import Corpus
from tqdm import tqdm

## Model

In [68]:
class SRNN_Softmax (nn.Module):
    def __init__(self, ntokens,nhid,ninp, device, n_layers=1, memory_size=104, memory_dim = 5):#replaced output_size parameter by ntokens
        super(SRNN_Softmax, self).__init__()
        self.ntokens = ntokens
        self.ninp = ninp
        self.n_layers = n_layers
        self.nhid = nhid
        self.device= device
        
        self.memory_size = memory_size
        self.memory_dim = memory_dim
        
        self.rnn = nn.RNN(self.ninp, self.nhid, self.n_layers)

        self.W_y = nn.Linear(self.nhid, ntokens)
        self.W_n = nn.Linear(self.nhid, self.memory_dim)
        self.W_a = nn.Linear(self.nhid, 2)
        self.W_sh = nn.Linear (self.memory_dim, self.nhid)
        self.encoder = nn.Embedding(self.ntokens, self.ninp)
        self.decoder = nn.Linear(ninp, ntokens)
        print(ntokens)
        print(ninp)
        # Actions -- push : 0 and pop: 1
        self.softmax = nn.Softmax(dim=2) 
        self.sigmoid = nn.Sigmoid ()
        self.softmax_out = nn.Softmax(dim=-1)
    
    def init_hidden (self, batch_size):
        return torch.zeros (self.n_layers, batch_size, self.nhid).to(self.device)#additional modification to allow batch processing
    
    # def forward(self, input, hidden0, stack, temperature=1.):
    #     seq_len = input.size(0)
    #     batch_size = input.size(1)
    #     # input is [seq len, batch size]
    #     # hidden0 : [1, batch size, nhid]
    #     # stack : [memory size, memory dim] -> stack[0] : [memory dim]
    #     emb = self.encoder(input)#maybe add some dropout here in the future
    #     # emb is [seq len, batch size, ninp]
    #     hidden_bar = self.W_sh (stack[0]).view(1, 1, -1) + hidden0
    #     # W_sh maps stack[0] from [memory dim] to [nhid]
    #     # the view operator give [1,1,nhid] -> hidden_bar is [1, batch_size, nhid]]
    #     ht, hidden = self.rnn(emb, hidden_bar)
    #     # ht : [seq len, batch size, nhid]
    #     # hidden : [n_layers, batch size, nhid]
    #     output = self.softmax_out(self.sigmoid(self.W_y(ht)).view(-1, self.ntokens))
    #     # W_y maps ht from [seq len, batch size, nhid] to [seq len, batch size, ntokens]. Then view operator gives [seq len*batch size, ntokens]
    #     # Then we apply sigmoid and softmax.
    #     # In Yair's adaptation, there's an embedding layer between sigmoid and softmax. But W_y already maps to the right dimensions, so why another layer ?
    #     self.action_weights = self.softmax (self.W_a (ht)).view(-1)
    #     # W_a maps ht from [seq len, batch size, nhid] to [seq len, batch size, 2] (proba of push and pop)
    #     # softmax is applied along the last dimension (2)
    #     # view operator gives [seq len * batchsize * 2]
    #     self.new_elt = self.sigmoid (self.W_n(ht))#.view(seq_len*batch_size, self.memory_dim)
    #     # Wn maps ht to [seq len, batch size, memory dim]
    #     # I NEED TO LOOP OVER THAT
    #     # view(1, self.memory_dim) operator suppose que seq len * batch size = 1. 
    #     # We get rid of it (we find the same shape later on with the unsqueeze(O) in push side)
    #     # We loop over sequence and batch.
    #     for i in range(self.new_elt.size(1)):#loop over elements in the batch
    #         for j in range(self.new_elt.size(0)):#loop over elements in each sequence
    #             push_side = torch.cat ((self.new_elt[j,i].unsqueeze(0), stack[:-1]), dim=0)
    #             pop_side = torch.cat ((stack[1:], torch.zeros(1, self.memory_dim).to(self.device)), dim=0)
    #             stack = self.action_weights [0] * push_side + self.action_weights [1] * pop_side
    #     return output, hidden, stack
    #sinon autant direct mettre la boucle au début et on copie colle juste la forward du code. Parce que là on update la stack une fois par batch (vs une fois par token normalement)
        
    def forward(self, input, hidden0, stack, temperature=1.):
        seq_len = input.size(0)
        batch_size = input.size(1)
        emb = self.encoder(input)
        for j in range(batch_size):
            for i in range(seq_len):
                hidden_bar = self.W_sh (stack[0]).view(1, 1, -1) + hidden0
                ht, hidden = self.rnn(emb[i,j], hidden_bar)
                print('ht', ht.shape)
                output = self.sigmoid(self.W_y(ht)).view(-1, self.output_size)
                self.action_weights = self.softmax (self.W_a (ht)).view(-1)
                self.new_elt = self.sigmoid (self.W_n(ht)).view(1, self.memory_dim)
                push_side = torch.cat ((self.new_elt, stack[:-1]), dim=0)
                pop_side = torch.cat ((stack[1:], torch.zeros(1, self.memory_dim).to(device)), dim=0)
                stack = self.action_weights [0] * push_side + self.action_weights [1] * pop_side
        return output, hidden, stack

## Training function

In [ ]:
def train(model, criterion, train_data, batch_size, ntokens, memory_size, memory_dim, device, nheads=False):
    # Turn on training mode which enables dropout.
    model.train()
    total_loss = 0
    #NEW : move hidden to devide
    
    
    for batch, i in enumerate(tqdm(range(0, train_data.size(0) - 1, 35), desc="Training")):
        data, targets = get_batch(train_data, i, 35)
        #NEW : move data and target to device
        model.zero_grad()
        hidden = model.init_hidden(batch_size)
        memory = torch.zeros ( memory_size, memory_dim).to(device)        # truncated BPP
        
        output, hidden, memory = model(data, hidden,memory)
        print(output.shape)
        break
        # output_flat = output.reshape(-1, output.size(-1))
        
        # # Similarly, reshape targets to [seq_len*batch_size]
        # targets_flat = targets.reshape(-1)
        # #loss = criterion(output.view(-1, ntokens), targets)
        # loss=criterion(output_flat, targets_flat)
        # loss.backward()

        # # `clip_grad_norm` helps prevent the exploding gradient problem in RNNs / LSTMs.
        # torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
        # for p in model.parameters():
        #     p.data.add_(-10, p.grad.data)

        # total_loss += loss.item()

## Data

In [5]:
corpus = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')
ntokens = len(corpus.dictionary)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_data = batchify(corpus.train, 128, device)#batch size of 128
val_data = batchify(corpus.valid, 128, device)
test_data = batchify(corpus.test, 128, device)
criterion = nn.CrossEntropyLoss()


In [18]:
ntokens

50001

In [69]:
model = SRNN_Softmax(ntokens, 256, 200, device)

50001
200


In [70]:
#ninp : dimension of the embedding = 200
#nhid : dimension of the hidden layer = 256

In [71]:
train(model, criterion, train_data, 128, ntokens, 104, 5, device)#memory size and memory dim by default in the github


Training:   0%|          | 0/18540 [00:00<?, ?it/s]


AssertionError: RNN: Expected input to be 2-D or 3-D but received 1-D tensor

In [30]:
35*128*2

8960